In [25]:
from pyspark.sql import SparkSession 

from pyspark.sql.functions import *

from pyspark.sql.types import*

spark=SparkSession.builder.appName("Spark SQL Example").master("local[*]").getOrCreate()

#### Q1
Find employees earning more than their manager.

emp_id	name	salary	manager_id	dept_id
1	Alice	90000	3	D1
2	Bob	75000	3	D1
3	Carol	80000	NULL	D1
4	Dave	120000	5	D2
5	Eve	100000	NULL	D2
6	Frank	55000	3	D1

In [3]:
data = [
        (1, "Alice", 90000, 3,'D1'),
       (2, "Bob", 75000, 3,'D1'),
        (3, "Carol", 80000, None,'D1'),
        (4,"Dave",120000,5,'D2'),
        (5,"Eve",100000,None,'D2'),
        (6,"Frank",55000,3,'D1')
    
]

columns = ["emp_id", "name", "salary", "manager_id","dept_id"]


# create DataFrame

df = spark.createDataFrame(data, columns)
df.show()


+------+-----+------+----------+-------+
|emp_id| name|salary|manager_id|dept_id|
+------+-----+------+----------+-------+
|     1|Alice| 90000|         3|     D1|
|     2|  Bob| 75000|         3|     D1|
|     3|Carol| 80000|      NULL|     D1|
|     4| Dave|120000|         5|     D2|
|     5|  Eve|100000|      NULL|     D2|
|     6|Frank| 55000|         3|     D1|
+------+-----+------+----------+-------+



Alice manager is Carol and his salary is 80k and alice salary is 90k
Dave manager is eve and whoose salary is 100k and dave salary is 120k



our result will be 

Alice 
Carol 





In [4]:
df.createOrReplaceTempView("employees")



In [17]:
spark.sql("""
              select distinct e.name
              from employees e
              join employees m
              on e.manager_id = m.emp_id
              where e.salary >m.salary
          """).show()

+-----+
| name|
+-----+
| Dave|
|Alice|
+-----+



####  how self join works 

 step 1: Identify relation what is asked in question 

     emp ---->manger 

     employee references manager

    '''Relationship rule''': Whenever one column stores another row’s ID:

    
    Employee -----> Manager

    Because question talks about:  employee compared with manager

    

    




   step2:   match foriegn key to primary key 

           in emp table fk will e.manager_id and m.emp_id 

                       reference → actual row

              

🔥 MASTER RULE

Join condition usually remains SAME:  e.manager_id = m.emp_id


Because relationship never changes:  employee stores manager_id



In [18]:
# lets try with dataframe api

df.show()



+------+-----+------+----------+-------+
|emp_id| name|salary|manager_id|dept_id|
+------+-----+------+----------+-------+
|     1|Alice| 90000|         3|     D1|
|     2|  Bob| 75000|         3|     D1|
|     3|Carol| 80000|      NULL|     D1|
|     4| Dave|120000|         5|     D2|
|     5|  Eve|100000|      NULL|     D2|
|     6|Frank| 55000|         3|     D1|
+------+-----+------+----------+-------+



In [24]:
# what we need to  we need to perform selef join then filter condition 


# as we are doing self join we need two differnet table / data frames to join 


emp_df =df.alias("e")

manger_df=df.alias("m")

# without alias spark will not understant which one is emp and manager 


# now we will perfrom join 


result=emp_df.join(manger_df,col("e.manager_id")== col("m.emp_id"))\
             .filter(col("e.salary")>col("m.salary"))



In [ ]:
result.show()

+------+-----+------+----------+-------+------+-----+------+----------+-------+
|emp_id| name|salary|manager_id|dept_id|emp_id| name|salary|manager_id|dept_id|
+------+-----+------+----------+-------+------+-----+------+----------+-------+
|     1|Alice| 90000|         3|     D1|     3|Carol| 80000|      NULL|     D1|
|     4| Dave|120000|         5|     D2|     5|  Eve|100000|      NULL|     D2|
+------+-----+------+----------+-------+------+-----+------+----------+-------+



In [26]:
result.select(
       col("e.emp_id").alias("emplyooe_id"),
       col("e.name").alias("emp_name")
).show()

+-----------+--------+
|emplyooe_id|emp_name|
+-----------+--------+
|          1|   Alice|
|          4|    Dave|
+-----------+--------+



## customer who placed no order 

customer table :

| cust_id | name  | city   |
| ------- | ----- | ------ |
| 1       | Priya | Delhi  |
| 2       | Ravi  | Mumbai |
| 3       | Sneha | Pune   |
| 4       | Arjun | Delhi  |

orders table :

| order_id | cust_id | amount |
| -------- | ------- | ------ |
| 101      | 1       | 500    |
| 102      | 1       | 300    |
| 103      | 2       | 1200   |
| 104      | 3       | 800    |

find customer who has not place any order 

so i need to select those cust_id which not in orders table 




In [2]:
# lets create dataframe first 

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Customer Data
customer_data = [
    (1, "Priya", "Delhi"),
    (2, "Ravi", "Mumbai"),
    (3, "Sneha", "Pune"),
    (4, "Arjun", "Delhi")
]

customer_columns = ["cust_id", "name", "city"]

customer_df = spark.createDataFrame(customer_data, customer_columns)

# Orders Data
orders_data = [
    (101, 1, 500),
    (102, 1, 300),
    (103, 2, 1200),
    (104, 3, 800)
]

orders_columns = ["order_id", "cust_id", "amount"]

orders_df = spark.createDataFrame(orders_data, orders_columns)

orders_df.show()

customer_df.show()

+--------+-------+------+
|order_id|cust_id|amount|
+--------+-------+------+
|     101|      1|   500|
|     102|      1|   300|
|     103|      2|  1200|
|     104|      3|   800|
+--------+-------+------+

+-------+-----+------+
|cust_id| name|  city|
+-------+-----+------+
|      1|Priya| Delhi|
|      2| Ravi|Mumbai|
|      3|Sneha|  Pune|
|      4|Arjun| Delhi|
+-------+-----+------+



In [9]:
# lets creat temp vieve on both table 

customer_df.createOrReplaceTempView("customer")

orders_df.createOrReplaceTempView("orders")

In [ ]:
spark.sql("""
    select name
    from customer
    where cust_id not in (
        select cust_id from orders
    )
""").show()

# query is correct but we can use here left jon and filter using null 

# WHERE cust_id NOT IN (1,2,NULL)

# SQL gets confused because comparison with NULL is unknown.

# result can be empty 

+-----+
| name|
+-----+
|Arjun|
+-----+



In [24]:
# lets see using left join 

spark.sql("""
              select c.name
              from customer c
              left join orders o
              on c.cust_id=o.cust_id
              where o.cust_id is null
          """).show()

+-----+
| name|
+-----+
|Arjun|
+-----+



In [30]:
# lest do using data frame api 

result_df=customer_df.join(orders_df,
                   # condition
                    customer_df.cust_id==orders_df.cust_id,
                    # type of join 
                    'left'
                 ).filter(orders_df.cust_id.isNull()).select(customer_df.name)


In [32]:
result_df.show()

+-----+
| name|
+-----+
|Arjun|
+-----+

